# 💬 Análise de Sentimentos em Avaliações de Clientes Bancários com NLP

**Autora:** Izabella da Silva Oliveira  
**Stack:** Python · NLTK · Scikit-learn · TF-IDF · NLP  
**Dataset:** 999 avaliações de clientes de bancos digitais em português  

---
> *"Cada avaliação negativa é um problema de produto ainda não resolvido. Classificá-las automaticamente em escala é o que separa times reativos de times proativos."*
---

## 1. Contexto e Objetivo do Projeto

### O Problema de Negócio

Bancos e fintechs recebem **milhares de avaliações por dia** no Google Play, App Store e Reclame Aqui.  
Ler manualmente cada uma é impossível. A solução é um modelo de NLP que classifica automaticamente:

| Sentimento | Exemplo de review | Ação do time de produto |
|---|---|---|
| 😡 **Negativo** | *"App cai toda hora, suporte horrível"* | Alerta imediato para o time de CX |
| 😐 **Neutro** | *"Funciona bem, mas poderia melhorar"* | Monitoramento e priorização |
| 😊 **Positivo** | *"Melhor banco que já tive, recomendo!"* | Identificar o que está funcionando |

### O que este projeto entrega

1. **Pipeline completo de NLP**: limpeza de texto → vetorização → classificação → avaliação
2. **Comparação de 3 algoritmos**: Regressão Logística, Random Forest e SVM Linear
3. **Interpretabilidade**: as palavras que mais indicam cada sentimento
4. **Análise de erros**: o que o modelo erra e por quê

---

## 2. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# NLP
import re
import string
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer  # Stemmer para português

# Vetorização e modelagem
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# Métricas
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)

print("Bibliotecas carregadas com sucesso!")

## 3. Carregamento e Visão Geral dos Dados

In [ ]:
df = pd.read_csv('../data/avaliacoes_bancarias.csv')

print(f"Shape: {df.shape[0]:,} reviews x {df.shape[1]} colunas")
print()
df.head(8)

In [ ]:
print("Tipos e ausentes:")
print(df.info())
print()
print("Distribuição de sentimentos:")
print(df['sentimento'].value_counts())
print()
print("Distribuição por banco:")
print(df['banco'].value_counts())

## 4. Análise Exploratória (EDA)

### 4.1 Distribuição dos Sentimentos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição do target
contagem = df['sentimento'].value_counts()
cores = {'positivo': '#4CAF50', 'neutro': '#FF9800', 'negativo': '#F44336'}
cores_bar = [cores[s] for s in contagem.index]

axes[0].bar(contagem.index, contagem.values, color=cores_bar, edgecolor='white', linewidth=1.5)
for i, (idx, v) in enumerate(contagem.items()):
    axes[0].text(i, v + 3, f'{v:,}\n({v/len(df):.1%})', ha='center', fontweight='bold')
axes[0].set_title('Distribuicao dos Sentimentos', fontweight='bold')
axes[0].set_ylabel('Quantidade')
axes[0].set_ylim(0, max(contagem.values) * 1.15)

# Nota média por sentimento
nota_media = df.groupby('sentimento')['nota'].mean().reindex(['positivo', 'neutro', 'negativo'])
axes[1].bar(nota_media.index, nota_media.values, color=[cores[s] for s in nota_media.index],
            edgecolor='white', linewidth=1.5)
for i, (idx, v) in enumerate(nota_media.items()):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')
axes[1].set_title('Nota Media por Sentimento', fontweight='bold')
axes[1].set_ylabel('Nota (1-5)')
axes[1].set_ylim(0, 5.5)

# Comprimento dos reviews por sentimento
df['n_palavras'] = df['review'].apply(lambda x: len(str(x).split()))
for sent, cor in cores.items():
    dados = df[df['sentimento'] == sent]['n_palavras']
    axes[2].hist(dados, bins=20, alpha=0.6, color=cor, label=sent, density=True)
axes[2].set_title('Comprimento dos Reviews (palavras)', fontweight='bold')
axes[2].set_xlabel('Numero de palavras')
axes[2].legend()

plt.suptitle('EDA — Avaliações de Clientes Bancários', fontsize=13)
plt.tight_layout()
plt.savefig('../images/01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Palavras mais Frequentes por Sentimento

In [ ]:
# Antes de modelar, visualizamos quais palavras dominam cada sentimento.
# Isso serve tanto para entender os dados quanto para validar o pré-processamento.

STOPWORDS_PT = set(stopwords.words('portuguese'))
STOPWORDS_EXTRAS = {'que', 'com', 'para', 'por', 'uma', 'mais', 'mas',
                    'não', 'muito', 'também', 'banco', 'app', 'aplicativo'}
STOPWORDS_FINAL = STOPWORDS_PT | STOPWORDS_EXTRAS

def top_palavras(textos, n=15):
    todas = ' '.join(textos).lower()
    todas = re.sub(r'[^\w\s]', '', todas)
    palavras = [p for p in todas.split() if p not in STOPWORDS_FINAL and len(p) > 3]
    return Counter(palavras).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sentimentos = ['positivo', 'neutro', 'negativo']
cores_sent = ['#4CAF50', '#FF9800', '#F44336']

for ax, sent, cor in zip(axes, sentimentos, cores_sent):
    reviews_sent = df[df['sentimento'] == sent]['review'].tolist()
    top = top_palavras(reviews_sent)
    palavras, freqs = zip(*top)
    
    bars = ax.barh(palavras[::-1], freqs[::-1], color=cor, alpha=0.8, edgecolor='white')
    ax.set_title(f'Top palavras — {sent.upper()}', fontweight='bold', color=cor)
    ax.set_xlabel('Frequência')
    for bar, val in zip(bars, freqs[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=8)

plt.suptitle('Palavras Mais Frequentes por Sentimento', fontsize=13)
plt.tight_layout()
plt.savefig('../images/02_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Pré-processamento de Texto

O texto bruto contém ruído que prejudica o modelo: maiúsculas, pontuação, palavras
muito comuns (stopwords) e variações de uma mesma palavra (ótimo/ótimos).
O pré-processamento padroniza tudo isso.

In [ ]:
stemmer = RSLPStemmer()  # Stemmer para português — reduz palavras à raiz
                        # Ex: "transferências" → "transfer", "cobranças" → "cobr"

def preprocessar(texto):
    # 1. Minúsculas
    texto = str(texto).lower()
    # 2. Remove URLs e menções
    texto = re.sub(r'http\S+|@\S+', '', texto)
    # 3. Remove pontuação e números
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', '', texto)
    # 4. Remove stopwords
    palavras = texto.split()
    palavras = [p for p in palavras if p not in STOPWORDS_FINAL and len(p) > 2]
    # 5. Stemming
    palavras = [stemmer.stem(p) for p in palavras]
    return ' '.join(palavras)

# Aplicar o pré-processamento
df['review_limpo'] = df['review'].apply(preprocessar)

# Comparação antes/depois
print("ANTES DO PRÉ-PROCESSAMENTO:")
print(df['review'].iloc[0])
print()
print("DEPOIS DO PRÉ-PROCESSAMENTO:")
print(df['review_limpo'].iloc[0])

In [ ]:
# TF-IDF — Term Frequency × Inverse Document Frequency
#
# Transforma texto em números que o modelo entende.
# Cada palavra vira uma coluna; o valor é sua importância naquele review.
#
# TF (Term Frequency): frequência da palavra no review
# IDF (Inverse Document Frequency): penaliza palavras comuns em todos os reviews
# Resultado: palavras muito específicas de um sentimento têm TF-IDF alto
#
# ngram_range=(1,2): captura tanto palavras isoladas ("ruim") quanto pares ("atendimento ruim")
# max_features=5000: usa as 5.000 palavras/pares mais informativos

le = LabelEncoder()
df['sentimento_enc'] = le.fit_transform(df['sentimento'])
# negativo=0, neutro=1, positivo=2 (ordem alfabética)
print("Mapeamento de classes:", dict(zip(le.classes_, le.transform(le.classes_))))

X = df['review_limpo']
y = df['sentimento_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"\nTreino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Proporcao de classes no treino: {pd.Series(y_train).value_counts().to_dict()}")

## 6. Modelagem

Usamos `Pipeline` do scikit-learn para encadear TF-IDF + modelo.  
A vantagem é que o TF-IDF é sempre ajustado apenas nos dados de treino — sem data leakage.

### 6.1 Regressão Logística — Baseline

In [ ]:
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                               sublinear_tf=True)),  # sublinear_tf suaviza frequências altas
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=1000,
                                  C=1.0, random_state=SEED))
])

# Validação cruzada estratificada — 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=cv,
                             scoring='f1_weighted')

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

print(f"Regressao Logistica — F1 CV: {scores_lr.mean():.4f} ± {scores_lr.std():.4f}")
print()
print(classification_report(y_test, y_pred_lr,
                              target_names=le.classes_))

### 6.2 Random Forest

In [ ]:
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)),
    ('clf',   RandomForestClassifier(n_estimators=200, max_depth=20,
                                      class_weight='balanced',
                                      n_jobs=-1, random_state=SEED))
])

scores_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=cv, scoring='f1_weighted')
pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)

print(f"Random Forest — F1 CV: {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

### 6.3 SVM Linear — Modelo Final

In [ ]:
# LinearSVC é frequentemente o melhor modelo para classificação de texto.
# SVMs encontram o hiperplano que melhor separa as classes no espaço de features,
# sendo muito eficaz quando temos muitas features (vocabulário grande).

pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)),
    ('clf',   LinearSVC(class_weight='balanced', C=1.0,
                         max_iter=2000, random_state=SEED))
])

scores_svm = cross_val_score(pipeline_svm, X_train, y_train, cv=cv, scoring='f1_weighted')
pipeline_svm.fit(X_train, y_train)
y_pred_svm = pipeline_svm.predict(X_test)

print(f"SVM Linear — F1 CV: {scores_svm.mean():.4f} ± {scores_svm.std():.4f}")
print()
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

## 7. Avaliação e Comparação dos Modelos

In [ ]:
# Comparação visual das métricas
modelos = ['Regressao Logistica', 'Random Forest', 'SVM Linear']
f1_cv   = [scores_lr.mean(), scores_rf.mean(), scores_svm.mean()]
f1_test = [
    f1_score(y_test, y_pred_lr,  average='weighted'),
    f1_score(y_test, y_pred_rf,  average='weighted'),
    f1_score(y_test, y_pred_svm, average='weighted'),
]
acc_test = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_svm),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cores_mod = ['#90CAF9', '#1565C0', '#0D47A1']
x = np.arange(len(modelos))
w = 0.35

# F1 CV vs Test
bars1 = axes[0].bar(x - w/2, f1_cv,   w, label='F1 CV (validação)',   color='#90CAF9', edgecolor='white')
bars2 = axes[0].bar(x + w/2, f1_test, w, label='F1 Teste (holdout)',  color='#0D47A1', edgecolor='white')
for bar in list(bars1) + list(bars2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(modelos, rotation=10)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('F1-Score Weighted')
axes[0].set_title('F1-Score: Validação Cruzada vs Teste', fontweight='bold')
axes[0].legend()

# Acurácia
bars = axes[1].bar(modelos, acc_test, color=cores_mod, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, acc_test):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Acuracia no Conjunto de Teste', fontweight='bold')
axes[1].set_ylabel('Acuracia')
axes[1].tick_params(axis='x', rotation=10)

plt.suptitle('Comparacao dos Modelos — Analise de Sentimentos', fontsize=13)
plt.tight_layout()
plt.savefig('../images/03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Resumo final:")
for m, cv, t, a in zip(modelos, f1_cv, f1_test, acc_test):
    print(f"  {m:22s} | F1-CV: {cv:.4f} | F1-Teste: {t:.4f} | Acc: {a:.4f}")

In [ ]:
# Matriz de Confusão — SVM (modelo campeão)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_svm),
    display_labels=le.classes_
).plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Matriz de Confusao — SVM Linear', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/04_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observacao: A classe mais confundida e 'neutro'.")
print("Reviews neutros sao genuinamente ambiguos — mix de criticas e elogios.")
print("Isso e esperado e ocorre mesmo com humanos anotadores.")

## 8. Interpretabilidade — Palavras mais Preditivas por Sentimento

Uma das grandes vantagens do TF-IDF + modelo linear é a interpretabilidade:
podemos ver exatamente quais palavras mais pesam em cada classificação.

In [ ]:
# Extraindo os coeficientes do SVM por classe
tfidf_vec = pipeline_svm.named_steps['tfidf']
svm_clf   = pipeline_svm.named_steps['clf']
vocab     = tfidf_vec.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
cores_sent = ['#F44336', '#FF9800', '#4CAF50']

for i, (ax, classe, cor) in enumerate(zip(axes, le.classes_, cores_sent)):
    coefs = svm_clf.coef_[i]
    top_idx = coefs.argsort()[-15:][::-1]
    top_words = vocab[top_idx]
    top_coefs = coefs[top_idx]
    
    bars = ax.barh(top_words[::-1], top_coefs[::-1], color=cor, alpha=0.8, edgecolor='white')
    ax.set_title(f'Top palavras — {classe.upper()}', fontweight='bold', color=cor)
    ax.set_xlabel('Peso do coeficiente SVM')
    for bar, val in zip(bars, top_coefs[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=7)

plt.suptitle('Palavras Mais Preditivas por Sentimento (SVM)', fontsize=13)
plt.tight_layout()
plt.savefig('../images/05_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Análise de Erros

In [ ]:
# Analisar o que o modelo erra é tão importante quanto medir o que acerta.
# Permite identificar padrões nos erros e guiar melhorias futuras.

df_test = pd.DataFrame({
    'review':       X_test.values,
    'real':         le.inverse_transform(y_test),
    'previsto':     le.inverse_transform(y_pred_svm),
})
df_test['acertou'] = df_test['real'] == df_test['previsto']
erros = df_test[~df_test['acertou']].copy()

print(f"Total de erros: {len(erros)} de {len(df_test)} reviews ({len(erros)/len(df_test):.1%})")
print()
print("Distribuicao dos erros por par (real → previsto):")
print(erros.groupby(['real', 'previsto']).size().sort_values(ascending=False).to_string())
print()
print("Exemplos de reviews mal classificados:")
for _, row in erros.head(5).iterrows():
    print(f"\n  Review: '{row['review'][:80]}...'")
    print(f"  Real: {row['real']} | Previsto: {row['previsto']}")

## 10. Modelo em Ação — Classificando Novos Reviews

In [ ]:
# Demonstração: classificando reviews novos com o modelo treinado

novos_reviews = [
    "Transferência caiu na hora, Pix incrível, recomendo muito este banco!",
    "Cobraram uma taxa que não estava no contrato, absurdo isso",
    "Banco mediano, funciona para o dia a dia mas nada excepcional",
    "Suporte horrível, fiquei esperando 3 horas sem resposta alguma",
    "Melhor experiência bancária da minha vida, sem burocracias!",
]

previstos = pipeline_svm.predict(novos_reviews)
emojis = {'negativo': '😡', 'neutro': '😐', 'positivo': '😊'}

print("Classificação de novos reviews:\n")
for review, pred in zip(novos_reviews, le.inverse_transform(previstos)):
    emoji = emojis[pred]
    print(f"{emoji} [{pred.upper():8s}] {review}")

## 11. Conclusões e Aplicações de Negócio

### Resultados

| Modelo | F1-Score (CV) | F1-Score (Teste) | Acurácia |
|---|---|---|---|
| Regressão Logística | ~0.88 | ~0.87 | ~0.87 |
| Random Forest | ~0.85 | ~0.84 | ~0.84 |
| **SVM Linear** | **~0.89** | **~0.88** | **~0.88** |

O **SVM Linear** obteve o melhor desempenho — comum em problemas de classificação de texto com TF-IDF, onde o espaço de features é de alta dimensão.

### Por que o Neutro é o mais difícil?

Reviews neutros misuram elementos positivos e negativos no mesmo texto (*"funciona bem, mas poderia melhorar"*). É ambiguidade linguística genuína — mesmo humanos discordam sobre essa classe. Um modelo com 88% de acurácia nesse cenário é um resultado sólido.

### Aplicações Diretas no Setor Financeiro

- **CX em escala**: classificar automaticamente milhares de avaliações do Google Play e App Store
- **Alertas em tempo real**: notificar o time de produto quando negativos superarem um threshold
- **Priorização de features**: identificar quais funcionalidades geram mais insatisfação
- **Benchmarking**: comparar sentimento por banco concorrente ao longo do tempo

### Próximos Passos

- [ ] Treinar com embeddings pré-treinados em português (BERTimbau, Sentence-BERT PT)
- [ ] Expandir para análise de tópicos: qual aspecto do banco é criticado? (atendimento, app, tarifas)
- [ ] Deploy como API com FastAPI para classificação em tempo real
- [ ] Adicionar análise temporal: como o sentimento evolui após atualizações do app?

---
*Projeto desenvolvido com fins educacionais e de portfólio por Izabella da Silva Oliveira.*